In [2]:
# Run once
!pip install datasets transformers torch python-dotenv

  Obtaining dependency information for python-dotenv from https://files.pythonhosted.org/packages/0b/d7/1959b9648791274998a9c3526f6d0ec8fd2233e4d4acce81bbae76b44b2a/python_dotenv-1.2.2-py3-none-any.whl.metadata



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from dotenv import load_dotenv
import os
from huggingface_hub import login

# Load token from .env (SAFE)
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

# Login
login(token=hf_token)

In [5]:
from datasets import load_dataset

# Load your dataset (already downloaded)
dataset = load_dataset("weerayut/multilexnorm2026-dev-pub")

# Check data
print("Train rows:", len(dataset["train"]))
print("Languages:", set(dataset["train"]["lang"]))

Train rows: 39178
Languages: {'th', 'it', 'da', 'ko', 'tr', 'en', 'trde', 'sl', 'vi', 'hr', 'sr', 'de', 'id', 'es', 'ja', 'nl', 'iden'}


In [7]:
from collections import defaultdict

# MFR Functions (from official repo)
def counting(data):
    counts = defaultdict(lambda: defaultdict(int))
    for item in data:
        for raw, norm in zip(item["raw"], item["norm"]):
            counts[raw][norm] += 1
    return counts

def mfr(raw_tokens, counts):
    rules = {
        "u": "you",
        "r": "are",
        "ur": "your",
        "bc": "because",
        "bcuz": "because",
        "wut": "what",
        "k": "tidak",
        "gk": "tidak",
        "gak": "tidak"
    }
    result = []
    for t in raw_tokens:
        if t in rules:
            result.append(rules[t])
        elif t in counts:
            result.append(max(counts[t], key=counts[t].get))
        else:
            result.append(t)
    return result

def evaluate(raw_list, gold_list, pred_list):
    tp = fp = fn = 0
    for raw, gold, pred in zip(raw_list, gold_list, pred_list):
        for r, g, p in zip(raw, gold, pred):
            if r == g:
                if p == g:
                    tp += 1
                else:
                    fp += 1
            else:
                if p == g:
                    tp += 1
                else:
                    fn += 1
    total = tp + fp + fn
    err = (fp + fn) / total if total > 0 else 0
    print(f"ERR: {err:.4f} | TP: {tp} | FP: {fp} | FN: {fn}")
    return err

In [8]:
# Train on full data
train = dataset["train"]
val = dataset["validation"]

counts = counting(train)

# Test
print("Smoke Test:")
print(mfr(["because", "u", "r", "funny"], counts))

Smoke Test:
['because', 'you', 'are', 'funny']


In [9]:
import random

def show_random(lang=None):
    data = val
    if lang:
        data = val.filter(lambda x: x["lang"] == lang)
    idx = random.randint(0, len(data)-1)
    noisy = data[idx]["raw"]
    clean = data[idx]["norm"]
    pred = mfr(noisy, counts)
    
    print("="*50)
    print(f"LANG: {data[idx]['lang']}")
    print("NOISY:", noisy)
    print("REAL  :", clean)
    print("PRED  :", pred)
    print("="*50)

# Show 5 random examples
for _ in range(5):
    show_random()

LANG: sl
NOISY: ['sej', 'ponavadi', 'pojem', '3', 'kose', 'in', 'je', 'preveč', '.']
REAL  : ['saj', 'ponavadi', 'pojem', '3', 'kose', 'in', 'je', 'preveč', '.']
PRED  : ['saj', 'ponavadi', 'pojem', '3', 'kose', 'in', 'je', 'preveč', '.']
LANG: id
NOISY: ['Iiihh', 'papanya', 'lucu', 'yah', '@sitiginting_']
REAL  : ['ih', 'papanya', 'lucu', 'ya', '[mention]']
PRED  : ['Iiihh', 'papanya', 'lucu', 'ya', '@sitiginting_']
LANG: sr
NOISY: ['sve', 'dok', 'imam', 'ovakve', 'prijatelje', ',', 'ja', 'sam', 'bogata', ',', 'i', 'nikad', 'sama', '!']
REAL  : ['sve', 'dok', 'imam', 'ovakve', 'prijatelje', ',', 'ja', 'sam', 'bogata', ',', 'i', 'nikad', 'sama', '!']
PRED  : ['sve', 'dok', 'imam', 'ovakve', 'prijatelje', ',', 'ja', 'sam', 'bogata', ',', 'i', 'nikad', 'sama', '!']
LANG: de
NOISY: ['Ich', 'kenne', 'nichts', '(', 'das', 'so', 'schoen', 'ist', 'wie', 'du', ')']
REAL  : ['Ich', 'kenne', 'nichts', '(', 'das', 'so', 'schön', 'ist', 'wie', 'du', ')']
PRED  : ['Ich', 'kenne', 'nichts', '(', 'da

In [10]:
# Evaluate English
print("English:")
en_val = val.filter(lambda x: x["lang"] == "en")
preds = [mfr(x, counts) for x in en_val["raw"]]
evaluate(en_val["raw"], en_val["norm"], preds)

# Evaluate Indonesian
print("\nIndonesian:")
id_val = val.filter(lambda x: x["lang"] == "id")
preds = [mfr(x, counts) for x in id_val["raw"]]
evaluate(id_val["raw"], id_val["norm"], preds)

English:


Filter:   0%|          | 0/8408 [00:00<?, ? examples/s]

ERR: 0.0420 | TP: 8784 | FP: 132 | FN: 253

Indonesian:


Filter:   0%|          | 0/8408 [00:00<?, ? examples/s]

ERR: 0.2490 | TP: 3234 | FP: 27 | FN: 1045


0.24895494658615885

In [9]:
# ============================================
# COMPLETE CODE FOR ADVANCED MFR WITH CONTEXT
# ============================================

# Install required packages (run once)
!pip install datasets transformers torch python-dotenv

from dotenv import load_dotenv
import os
from huggingface_hub import login
from collections import defaultdict, Counter
import random

# Load token from .env (SAFE)
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
# Login
login(token=hf_token)

from datasets import load_dataset

# Load your dataset (already downloaded)
dataset = load_dataset("weerayut/multilexnorm2026-dev-pub")

# Check data
print("Train rows:", len(dataset["train"]))
print("Languages:", set(dataset["train"]["lang"]))

# ============================================
# MFR FUNCTIONS (FROM OFFICIAL REPO)
# ============================================

def counting(data):
    """Build counts dictionary for Most Frequent Replacement"""
    counts = defaultdict(lambda: defaultdict(int))
    for item in data:
        for raw, norm in zip(item["raw"], item["norm"]):
            counts[raw][norm] += 1
    return counts

def mfr(raw_tokens, counts, rules=None):
    """Most Frequent Replacement with fallback rules"""
    if rules is None:
        # Default rules for common English and Indonesian tokens
        rules = {
            "u": "you",
            "r": "are",  # Fixed: "r" should be "are" for English
            "ur": "your",
            "bc": "because",
            "bcuz": "because",
            "wut": "what",
            "k": "tidak",
            "gk": "tidak",
            "gak": "tidak",
            "2": "to",
            "4": "for",
            "b": "be",
            "c": "see",
            "y": "why",
            "n": "and",
            "pls": "please",
            "thx": "thanks"
        }
    
    result = []
    for t in raw_tokens:
        if t in rules:
            result.append(rules[t])
        elif t in counts:
            # Get most frequent normalization
            best_norm = max(counts[t], key=counts[t].get)
            result.append(best_norm)
        else:
            result.append(t)
    return result

def evaluate(raw_list, gold_list, pred_list):
    """Calculate ERR (Error Rate) metric"""
    tp = fp = fn = 0
    
    for raw, gold, pred in zip(raw_list, gold_list, pred_list):
        for r, g, p in zip(raw, gold, pred):
            if r == g:  # No normalization needed
                if p == g:  # System correctly didn't normalize
                    tp += 1
                else:       # System incorrectly normalized
                    fp += 1
            else:           # Normalization needed
                if p == g:  # System correctly normalized
                    tp += 1
                else:       # System failed to normalize correctly
                    fn += 1
    
    total = tp + fp + fn
    err = (fp + fn) / total if total > 0 else 0
    print(f"ERR: {err:.4f} | TP: {tp} | FP: {fp} | FN: {fn}")
    return err

# ============================================
# IMPROVED ADVANCED MFR WITH CONTEXT
# ============================================

def counting_advanced(data, ngram=2):
    """
    Build n-gram aware counts dictionary
    ngram=2: bigram (prev_token, current_token)
    ngram=3: trigram (prev_token, current_token, next_token)
    """
    counts = {
        "unigram": defaultdict(lambda: defaultdict(int)),
        f"{ngram}gram": defaultdict(lambda: defaultdict(int))
    }
    
    for item in data:
        raw_tokens = item["raw"]
        norm_tokens = item["norm"]
        
        for i in range(len(raw_tokens)):
            # Store unigram counts
            counts["unigram"][raw_tokens[i]][norm_tokens[i]] += 1
            
            # Store n-gram counts
            if ngram == 2:
                prev_token = raw_tokens[i-1] if i > 0 else "<s>"
                context_key = (prev_token, raw_tokens[i])
            elif ngram == 3:
                prev_token = raw_tokens[i-1] if i > 0 else "<s>"
                next_token = raw_tokens[i+1] if i < len(raw_tokens)-1 else "</s>"
                context_key = (prev_token, raw_tokens[i], next_token)
            else:
                context_key = raw_tokens[i]
            
            counts[f"{ngram}gram"][context_key][norm_tokens[i]] += 1
    
    return counts

def mfr_advanced(raw_tokens, counts, ngram=2, rules=None):
    """
    Improved MFR with n-gram context and intelligent fallback
    """
    if rules is None:
        rules = {
            "u": "you",
            "r": "are",  # Fixed: "r" should be "are" for English
            "ur": "your",
            "bc": "because",
            "bcuz": "because",
            "wut": "what",
            "k": "tidak",
            "gk": "tidak",
            "gak": "tidak",
            "2": "to",
            "4": "for"
        }
    
    result = []
    
    for i, token in enumerate(raw_tokens):
        # Step 1: Check hard-coded rules first
        if token in rules:
            result.append(rules[token])
            continue
        
        # Step 2: Try n-gram context matching
        if ngram == 2:
            prev_token = raw_tokens[i-1] if i > 0 else "<s>"
            context_key = (prev_token, token)
        elif ngram == 3:
            prev_token = raw_tokens[i-1] if i > 0 else "<s>"
            next_token = raw_tokens[i+1] if i < len(raw_tokens)-1 else "</s>"
            context_key = (prev_token, token, next_token)
        else:
            context_key = token
        
        ngram_counts = counts.get(f"{ngram}gram", {})
        
        if context_key in ngram_counts:
            # Get the most frequent mapping for this context
            best_norm = max(ngram_counts[context_key], 
                           key=ngram_counts[context_key].get)
            result.append(best_norm)
            continue
        
        # Step 3: Fallback to unigram
        unigram_counts = counts.get("unigram", {})
        if token in unigram_counts:
            best_norm = max(unigram_counts[token], 
                           key=unigram_counts[token].get)
            result.append(best_norm)
            continue
        
        # Step 4: Final fallback - keep original
        result.append(token)
    
    return result

# ============================================
# MAIN EXECUTION
# ============================================

# Prepare data
train = dataset['train']
val = dataset['validation']

# Build original counts
print("\nBuilding original MFR counts...")
original_counts = counting(train)

# Build advanced counts
print("Building advanced MFR counts (bigram context)...")
advanced_counts = counting_advanced(train, ngram=2)

# Test with examples
print("\n" + "="*60)
print("TESTING MFR MODELS")
print("="*60)

test_cases = [
    ["because", "u", "r", "funny"],
    ["i", "love", "u", "too"],
    ["gak", "bisa", "makan"],
    ["wut", "is", "this"],
    ["r", "u", "okay"]  # Test case for "r" normalization
]

print("\nTest cases with different models:")
for i, tokens in enumerate(test_cases):
    print(f"\nTest {i+1}: {tokens}")
    print(f"Original MFR:     {mfr(tokens, original_counts)}")
    print(f"Bigram MFR:       {mfr_advanced(tokens, advanced_counts, ngram=2)}")

# Show random examples
def show_random(lang=None, n=3):
    """Show random examples with predictions"""
    data = val
    if lang:
        data = val.filter(lambda x: x['lang'] == lang)
    
    for _ in range(n):
        idx = random.randint(0, len(data) - 1)
        noisy = data[idx]["raw"]
        clean = data[idx]["norm"]
        pred1 = mfr(noisy, original_counts)
        pred2 = mfr_advanced(noisy, advanced_counts, ngram=2)
        
        print("\n" + "="*50)
        print(f"LANG: {data[idx]['lang']}")
        print(f"NOISY: {noisy}")
        print(f"REAL:  {clean}")
        print(f"MFR:   {pred1}")
        print(f"BIGRAM: {pred2}")
        print("="*50)

print("\n" + "="*60)
print("RANDOM VALIDATION EXAMPLES")
print("="*60)
show_random()

# Evaluate on English
print("\n" + "="*60)
print("EVALUATION ON ENGLISH VALIDATION SET")
print("="*60)

en_val = val.filter(lambda x: x['lang'] == "en")

# 1. Original MFR
print("\n1. Original MFR (Unigram):")
preds_original = [mfr(x, original_counts) for x in en_val["raw"]]
err_original = evaluate(en_val["raw"], en_val["norm"], preds_original)

# 2. Bigram MFR
print("\n2. Bigram Context MFR:")
preds_bigram = [mfr_advanced(x, advanced_counts, ngram=2) for x in en_val["raw"]]
err_bigram = evaluate(en_val["raw"], en_val["norm"], preds_bigram)

# Compare performance
print("\n" + "="*60)
print("PERFORMANCE SUMMARY")
print("="*60)
print(f"Original MFR Error Rate: {err_original:.4f}")
print(f"Bigram MFR Error Rate:   {err_bigram:.4f}")

if err_bigram < err_original:
    improvement = ((err_original - err_bigram) / err_original) * 100
    print(f"Improvement: {improvement:.1f}% (Bigram is better)")
elif err_bigram > err_original:
    degradation = ((err_bigram - err_original) / err_original) * 100
    print(f"Degradation: {degradation:.1f}% (Original is better)")
else:
    print("No change in performance")

# Evaluate on Indonesian
print("\n" + "="*60)
print("EVALUATION ON INDONESIAN VALIDATION SET")
print("="*60)

id_val = val.filter(lambda x: x['lang'] == "id")

# 1. Original MFR
print("\n1. Original MFR (Unigram):")
preds_original_id = [mfr(x, original_counts) for x in id_val["raw"]]
err_original_id = evaluate(id_val["raw"], id_val["norm"], preds_original_id)

# 2. Bigram MFR
print("\n2. Bigram Context MFR:")
preds_bigram_id = [mfr_advanced(x, advanced_counts, ngram=2) for x in id_val["raw"]]
err_bigram_id = evaluate(id_val["raw"], id_val["norm"], preds_bigram_id)

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print("ENGLISH:")
print(f"  Original MFR: {err_original:.4f}")
print(f"  Bigram MFR:   {err_bigram:.4f}")
print("\nINDONESIAN:")
print(f"  Original MFR: {err_original_id:.4f}")
print(f"  Bigram MFR:   {err_bigram_id:.4f}")

# Show error analysis
print("\n" + "="*60)
print("ERROR ANALYSIS - Examples where models differ")
print("="*60)

error_examples = []
for i in range(min(10, len(en_val))):
    raw = en_val[i]["raw"]
    gold = en_val[i]["norm"]
    pred1 = mfr(raw, original_counts)
    pred2 = mfr_advanced(raw, advanced_counts, ngram=2)
    
    if pred1 != pred2:
        error_examples.append((i, raw, gold, pred1, pred2))

if error_examples:
    for idx, raw, gold, pred1, pred2 in error_examples[:5]:
        print(f"\nExample {idx}:")
        print(f"Raw:    {raw}")
        print(f"Gold:   {gold}")
        print(f"MFR:    {pred1}")
        print(f"Bigram: {pred2}")
else:
    print("No differences found between models in first 10 examples")

print("\n" + "="*60)
print("CODE EXECUTION COMPLETE")
print("="*60)


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Train rows: 39178
Languages: {'id', 'ko', 'sr', 'it', 'sl', 'nl', 'da', 'th', 'vi', 'iden', 'en', 'ja', 'tr', 'de', 'trde', 'es', 'hr'}

Building original MFR counts...
Building advanced MFR counts (bigram context)...

TESTING MFR MODELS

Test cases with different models:

Test 1: ['because', 'u', 'r', 'funny']
Original MFR:     ['because', 'you', 'are', 'funny']
Bigram MFR:       ['because', 'you', 'are', 'funny']

Test 2: ['i', 'love', 'u', 'too']
Original MFR:     ['i', 'love', 'you', 'too']
Bigram MFR:       ['i', 'love', 'you', 'too']

Test 3: ['gak', 'bisa', 'makan']
Original MFR:     ['tidak', 'bisa', 'makan']
Bigram MFR:       ['tidak', 'bisa', 'makan']

Test 4: ['wut', 'is', 'this']
Original MFR:     ['what', 'is', 'this']
Bigram MFR:       ['what', 'is', 'this']

Test 5: ['r', 'u', 'okay']
Original MFR:     ['are', 'you', 'okay']
Bigram MFR:       ['are', 'you', 'okay']

RANDOM VALIDATION EXAMPLES

LANG: sr
NOISY: ['ma', 'cao', ',', 'zdravo', 'vidimo', 'se', 'aprila']
REAL:  

In [10]:
# ================================================================
# COMPLETE TEXT NORMALIZATION PIPELINE WITH JSON EXPORT
# ================================================================
# Run once
!pip install datasets transformers torch python-dotenv

from dotenv import load_dotenv
import os
from huggingface_hub import login
from datasets import load_dataset
from collections import defaultdict, Counter
import json
import random

# Load token from .env
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
# Login
login(token=hf_token)

# Load your dataset
dataset = load_dataset("weerayut/multilexnorm2026-dev-pub")

# Check data
print("Train rows:", len(dataset["train"]))
print("Languages:", set(dataset["train"]["lang"]))

# Split dataset
train = dataset['train']
val = dataset['validation']

# ================================================================
# MFR FUNCTIONS
# ================================================================

def counting(data):
    """Build counts dictionary for Most Frequent Replacement"""
    counts = defaultdict(lambda: defaultdict(int))
    for item in data:
        for raw, norm in zip(item["raw"], item["norm"]):
            counts[raw][norm] += 1
    return counts

def mfr(raw_tokens, counts, rules=None):
    """Most Frequent Replacement with fallback rules"""
    if rules is None:
        # Default rules for common English and Indonesian tokens
        rules = {
            "u": "you",
            "r": "are",
            "ur": "your",
            "bc": "because",
            "bcuz": "because",
            "wut": "what",
            "k": "tidak",
            "gk": "tidak",
            "gak": "tidak",
            "2": "to",
            "4": "for",
            "b": "be",
            "c": "see",
            "y": "why",
            "n": "and",
            "pls": "please",
            "thx": "thanks"
        }
    
    result = []
    for t in raw_tokens:
        if t in rules:
            result.append(rules[t])
        elif t in counts:
            # Get most frequent normalization
            best_norm = max(counts[t], key=counts[t].get)
            result.append(best_norm)
        else:
            result.append(t)
    return result

def counting_advanced(data, ngram=2):
    """Build n-gram aware counts dictionary"""
    counts = {
        "unigram": defaultdict(lambda: defaultdict(int)),
        f"{ngram}gram": defaultdict(lambda: defaultdict(int))
    }
    
    for item in data:
        raw_tokens = item["raw"]
        norm_tokens = item["norm"]
        
        for i in range(len(raw_tokens)):
            # Store unigram counts
            counts["unigram"][raw_tokens[i]][norm_tokens[i]] += 1
            
            # Store n-gram counts
            if ngram == 2:
                prev_token = raw_tokens[i-1] if i > 0 else "<s>"
                context_key = (prev_token, raw_tokens[i])
            elif ngram == 3:
                prev_token = raw_tokens[i-1] if i > 0 else "<s>"
                next_token = raw_tokens[i+1] if i < len(raw_tokens)-1 else "</s>"
                context_key = (prev_token, raw_tokens[i], next_token)
            else:
                context_key = raw_tokens[i]
            
            counts[f"{ngram}gram"][context_key][norm_tokens[i]] += 1
    
    return counts

def mfr_advanced(raw_tokens, counts, ngram=2, rules=None):
    """Improved MFR with n-gram context and intelligent fallback"""
    if rules is None:
        rules = {
            "u": "you",
            "r": "are",
            "ur": "your",
            "bc": "because",
            "bcuz": "because",
            "wut": "what",
            "k": "tidak",
            "gk": "tidak",
            "gak": "tidak",
            "2": "to",
            "4": "for"
        }
    
    result = []
    for i, token in enumerate(raw_tokens):
        # Step 1: Check hard-coded rules first
        if token in rules:
            result.append(rules[token])
            continue
        
        # Step 2: Try n-gram context matching
        if ngram == 2:
            prev_token = raw_tokens[i-1] if i > 0 else "<s>"
            context_key = (prev_token, token)
        elif ngram == 3:
            prev_token = raw_tokens[i-1] if i > 0 else "<s>"
            next_token = raw_tokens[i+1] if i < len(raw_tokens)-1 else "</s>"
            context_key = (prev_token, token, next_token)
        else:
            context_key = token
        
        ngram_counts = counts.get(f"{ngram}gram", {})
        if context_key in ngram_counts:
            # Get the most frequent mapping for this context
            best_norm = max(ngram_counts[context_key],
                          key=ngram_counts[context_key].get)
            result.append(best_norm)
            continue
        
        # Step 3: Fallback to unigram
        unigram_counts = counts.get("unigram", {})
        if token in unigram_counts:
            best_norm = max(unigram_counts[token],
                          key=unigram_counts[token].get)
            result.append(best_norm)
            continue
        
        # Step 4: Final fallback - keep original
        result.append(token)
    
    return result

# ================================================================
# BUILD COUNTS
# ================================================================

print("\nBuilding original MFR counts...")
original_counts = counting(train)

print("Building advanced MFR counts (bigram context)...")
advanced_counts = counting_advanced(train, ngram=2)

# ================================================================
# EXPORT CLEANED TEXT TO JSON
# ================================================================

def export_cleaned_text(dataset_split, counts, model_type="advanced", 
                       output_filename="cleaned_text.json", ngram=2):
    """
    Export only cleaned text to JSON file
    - dataset_split: The dataset to process (train or val)
    - counts: The counts dictionary to use
    - model_type: "advanced" for mfr_advanced, "original" for mfr
    - output_filename: Name of JSON file to save
    - ngram: Context size for advanced model
    """
    
    cleaned_texts = []
    
    print(f"\nExporting cleaned text using {model_type} model...")
    print(f"Processing {len(dataset_split)} examples")
    
    for i in range(len(dataset_split)):
        raw_tokens = dataset_split[i]["raw"]
        
        if model_type == "advanced":
            cleaned = mfr_advanced(raw_tokens, counts, ngram=ngram)
        else:
            cleaned = mfr(raw_tokens, counts)
        
        cleaned_texts.append(cleaned)
        
        # Progress indicator
        if (i + 1) % 1000 == 0:
            print(f"  Processed {i + 1} examples")
    
    # Save to JSON file
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(cleaned_texts, f, indent=2, ensure_ascii=False)
    
    print(f"\n✅ Successfully exported {len(cleaned_texts)} cleaned texts to {output_filename}")
    print(f"   File contains ONLY the cleaned text arrays")
    print(f"   JSON structure: Array of arrays, e.g., [[word1, word2, ...], [...], ...]")
    
    return cleaned_texts

# ================================================================
# RUN EXPORT
# ================================================================

print("\n" + "="*60)
print("EXPORTING CLEANED TEXT TO JSON")
print("="*60)

# Export validation data with advanced (bigram) model
export_cleaned_text(
    dataset_split=val,
    counts=advanced_counts,
    model_type="advanced",
    output_filename="cleaned_validation_bigram.json",
    ngram=2
)

# Export training data with advanced (bigram) model
export_cleaned_text(
    dataset_split=train,
    counts=advanced_counts,
    model_type="advanced",
    output_filename="cleaned_train_bigram.json",
    ngram=2
)

# Export validation data with original model
export_cleaned_text(
    dataset_split=val,
    counts=original_counts,
    model_type="original",
    output_filename="cleaned_validation_original.json"
)

# ================================================================
# SHOW SAMPLE OUTPUT
# ================================================================

print("\n" + "="*60)
print("SAMPLE OF EXPORTED DATA")
print("="*60)

# Load and show a sample from the first file
with open("cleaned_validation_bigram.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"\nLoaded {len(data)} cleaned text arrays from cleaned_validation_bigram.json")
print("\nFirst 3 cleaned texts:")
for i, text in enumerate(data[:3]):
    print(f"[{i}]: {text}")

print("\n" + "="*60)
print("EXPORT COMPLETE")
print("="*60)
print("Created files:")
print("1. cleaned_validation_bigram.json  - Validation set with advanced model")
print("2. cleaned_train_bigram.json        - Training set with advanced model")
print("3. cleaned_validation_original.json - Validation set with original model")
print("\n" + "="*60)
print("HOW TO USE THE EXPORTED FILES:")
print("="*60)
print("```python")
print("import json")
print("# Load the cleaned text")
print("with open('cleaned_validation_bigram.json', 'r') as f:")
print("    cleaned_texts = json.load(f)")
print("# cleaned_texts is a list of word arrays")
print("print(f'Loaded {len(cleaned_texts)} cleaned texts')")
print("print('First cleaned text:', cleaned_texts[0])")
print("```")


[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Train rows: 39178
Languages: {'id', 'ko', 'sr', 'it', 'sl', 'nl', 'da', 'th', 'vi', 'iden', 'en', 'ja', 'tr', 'de', 'trde', 'es', 'hr'}

Building original MFR counts...
Building advanced MFR counts (bigram context)...

EXPORTING CLEANED TEXT TO JSON

Exporting cleaned text using advanced model...
Processing 8408 examples
  Processed 1000 examples
  Processed 2000 examples
  Processed 3000 examples
  Processed 4000 examples
  Processed 5000 examples
  Processed 6000 examples
  Processed 7000 examples
  Processed 8000 examples

✅ Successfully exported 8408 cleaned texts to cleaned_validation_bigram.json
   File contains ONLY the cleaned text arrays
   JSON structure: Array of arrays, e.g., [[word1, word2, ...], [...], ...]

Exporting cleaned text using advanced model...
Processing 39178 examples
  Processed 1000 examples
  Processed 2000 examples
  Processed 3000 examples
  Processed 4000 examples
  Processed 5000 examples
  Processed 6000 examples
  Processed 7000 examples
  Processed 8